In [1]:
import pandas as pd

from pfas_trace.config import DATASET_IMPORT

# Data Validation and Decisions
Here we document inconsistencies in the dataset.  
Because we focus only on France it is already limited to this country.

In [2]:
df = pd.read_parquet(DATASET_IMPORT).query('country=="France"')
df.shape

(1094220, 21)

In [3]:
df.sort_values(by=["date"], ascending=False).head(3)

,category,lat,lon,name,city,country,type,sector,source_type,data_collection_method,...,source_url,dataset_id,dataset_name,pfas_values,unit,pfas_sum,details,matrix,date,year
872199,Measurement,47.980380,-4.080169,QUIMPER - TROHEIR,QUIMPER,France,Sampling location,NaN,Authorities,Public data - Download,...,https://www.data.gouv.fr/fr/datasets/resultats...,124,france_eaurob,"[{""cas_id"": ""2706-90-3"", ""unit"": ""ng/l"", ""subs...",ng/l,1.0,"{""inae"": ""EMI"", ""referenceprel"": ""02900270460""...",Drinking water,2026-08-31,2026.0
872287,Measurement,43.788462,4.436405,GARONS BOUILLARGUES,BOUILLARGUES,France,Sampling location,NaN,Authorities,Public data - Download,...,https://www.data.gouv.fr/fr/datasets/resultats...,124,france_eaurob,"[{""cas_id"": ""307-24-4"", ""unit"": ""ng/l"", ""subst...",ng/l,23.0,"{""inae"": ""EMI"", ""referenceprel"": ""03000199435""...",Drinking water,2026-08-31,2026.0
873606,Measurement,49.232418,5.967725,MOUTIERS-TITANITE,MOUTIERS,France,Sampling location,NaN,Authorities,Public data - Download,...,https://www.data.gouv.fr/fr/datasets/resultats...,124,france_eaurob,"[{""cas_id"": ""1417782-03-6"", ""unit"": ""ng/l"", ""s...",ng/l,0.0,"{""inae"": ""EMI"", ""referenceprel"": ""05400179730""...",Drinking water,2026-08-31,2026.0


In [4]:
df.groupby("category").size()

category
Known PFAS user                  27
Measurement                 1092682
PFAS production facility          5
Presumptive                    1506
dtype: int64

In [16]:
df.groupby("dataset_name").size()

dataset_name
AFFF from France                        5
ANSES                                 450
CampagneNationale2018_PFAS             45
EPRTR                                 769
France - ICPE                       32484
Known PFAS Users                       27
Major Airports                        154
Military bases                         39
Ozon l'eau saine                      212
PFAS Producers                          5
PFAS_in_surface_water_2019-2020        97
Paper Mills                            85
Radio France - Drinking water          89
Rhine Aquifer                        1012
france_ades                        316245
france_eaurob                      404677
france_naiades                     337371
wwtp                                  454
dtype: int64

In [23]:
df.sort_values(by=["pfas_sum"], ascending=False)[
    ['pfas_sum', 'matrix', 'lat', 'lon', 'city', 'type', 'source_type', 'dataset_name']
].head(10)

,pfas_sum,matrix,lat,lon,city,type,source_type,dataset_name
1294135,509000000.0,Wastewater,43.380003,-0.603006,MOURENX,Sampling location,Authorities,France - ICPE
1320078,420000000.0,Wastewater,49.310362,1.007807,SAINT-AUBIN-LES-ELBEUF,Sampling location,Authorities,France - ICPE
1298274,91000000.0,Wastewater,44.171826,4.151853,SALINDRES,Sampling location,Authorities,France - ICPE
1319940,51000000.0,Wastewater,49.291294,2.507002,VILLERS-SAINT-PAUL,Sampling location,Authorities,France - ICPE
1294157,50000000.0,Wastewater,43.380003,-0.603006,MOURENX,Sampling location,Authorities,France - ICPE
1319920,49000000.0,Wastewater,49.291294,2.507002,VILLERS-SAINT-PAUL,Sampling location,Authorities,France - ICPE
1298271,28900000.0,Wastewater,44.171826,4.151853,SALINDRES,Sampling location,Authorities,France - ICPE
1320009,28000000.0,Surface water,49.306230,1.007527,SAINT-AUBIN-LES-ELBEUF,Sampling location,Authorities,France - ICPE
1319698,26700000.0,Wastewater,49.291294,2.507002,VILLERS-SAINT-PAUL,Sampling location,Authorities,France - ICPE
1298125,17500000.0,Surface water,44.169532,4.151732,SALINDRES,Sampling location,Authorities,France - ICPE


## Missing Dates
Just 99.85% of the records have valid dates.

**Decision**
- Because most of the missing date-records also dont have a year set we decided to drop all records without a proper date set.

In [5]:
df_nodate = df.query('date.isna()')
df_nodate_no_year = df_nodate.query('year.isna()')


print(f'{df_nodate.shape[0] / df.shape[0]:.2%} ({df_nodate.shape[0]}) of the records have no date set!')
print(f'{df_nodate_no_year.shape[0] / df_nodate.shape[0]:.2%} ({df_nodate_no_year.shape[0]}) of these records have also no year set!')

0.15% (1680) of the records have no date set!
91.55% (1538) of these records have also no year set!


## Missing Coordinates
~30% of the records are missing coordinatres.
A possibility could be to match city-name with OpenStreetmap and extract the missing coordinates. But this would reduce the accuracy of our distance features and might be difficult to unveil companies who do not exist anymore.

**Decision**
- To not increase the complexity before we haven't implemented the whole ML-process these records will just be dropped.

In [6]:
df_missing_coordinates = df.query('lat.isna() or lon.isna()')

All missing values are "measurements"

In [7]:
df_missing_coordinates.category.value_counts()

category
Measurement    358304
Name: count, dtype: int64

In [13]:
df_missing_coordinates.groupby("matrix").size()

matrix
Drinking water     69006
Groundwater       221009
Rainwater              8
Sea water            106
Surface water      59533
Unknown             8630
Wastewater            12
dtype: int64

In [8]:
df_missing_coordinates.source_type.value_counts()

source_type
Authorities                 358215
Journalism investigation        89
Name: count, dtype: int64

Missing values exist over 4 different datasets.

In [9]:
df_missing_coordinates.dataset_name.value_counts()

dataset_name
france_eaurob                    357685
ANSES                               450
Radio France - Drinking water        89
France - ICPE                        80
Name: count, dtype: int64

In [10]:
df_missing_coordinates.shape[0]

358304

In [11]:
df_missing_coordinates.shape[0] / df.shape[0]

0.32745151797627536

### Outside of France

Some measurements are taken outside of mainland France.

In [18]:
# Same rows as features.parquet: measurements with coordinates and a date
df_with_coordinates = df.query('category == "Measurement"').dropna(subset=["lat", "lon"])

# Rough bounding box around metropolitan France and Corsica (own choice, not an official boundary)
in_metropolitan_france = df_with_coordinates["lat"].between(41, 51.5) & df_with_coordinates["lon"].between(-5.5, 10)
df_outside = df_with_coordinates[~in_metropolitan_france]

# Locations that match no overseas territory: the Atlantic off Africa and the sea off Brittany
wrong_coordinates = df_outside["lat"].between(-10, 0) & df_outside["lon"].between(-5, 5) | (df_outside["lat"] > 40)
df_wrong_coordinates = df_outside[wrong_coordinates]

print(f'{df_outside.shape[0] / df_feature_rows.shape[0]:.2%} ({df_outside.shape[0]}) of the {df_feature_rows.shape[0]} feature rows lie outside metropolitan France!')
print(f'{df_outside.shape[0] - df_wrong_coordinates.shape[0]} of these lie in overseas territories')
print(f'{df_wrong_coordinates.shape[0]} of these have wrong coordinates')

1.69% (12392) of the 734236 feature rows lie outside metropolitan France!
12368 of these lie in overseas territories
24 of these have wrong coordinates
